In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, max, count, min, approx_count_distinct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

spark = (SparkSession.builder
         .appName("perform-aagregations")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

In [0]:
df = (spark.read.format("csv")
      .option("header", "true")
      .option("nullValue", "null")
      .option("dateFormat", "LLLL d, y")
      .load("../data/netflix_titles.csv"))

In [0]:
# Group the data by a column
grouped_df = df.groupBy("country")

In [0]:
# Count the number of rows in each group
count_df = grouped_df.count()
count_df.show()

In [0]:
# Apply custom aggregation using max
max_release_df = grouped_df.agg(max(col("date_added")))
max_release_df.show()

In [0]:
release_date_gouped_df = (
    df.groupBy("country")
    .agg(
        count("show_id").alias("NumberOfReleases")
        ,max("date_added").alias("LastReleaseDate")
        ,min("date_added").alias("FirstReleaseDate")))

release_date_gouped_df.show(3)

### Pivot Tables

In [0]:
pivot_table = (df.groupBy("country").pivot("type").agg(count("show_id")))
pivot_table.show()

### Approximate Aggregations

In [0]:
# Define a Schema
schema = StructType([
    StructField("Id", IntegerType(), True),
    StructField("ProductId", StringType(), True),
    StructField("UserId", StringType(), True),
    StructField("ProfileName", StringType(), True),
    StructField("HelpfulnessNumerator", StringType(), True),
    StructField("HelpfulnessDenominator", StringType(), True),
    StructField("Score", IntegerType(), True),
    StructField("Time", StringType(), True),
    StructField("Summary", StringType(), True),
    StructField("Text", StringType(), True)])


review_df = (spark.read.format("csv")
      .option("header",True)
      .schema(schema)
      .load("../data/Reviews.csv"))

In [0]:
# Approximate quantile calculation
quantiles = review_df.approxQuantile("Score", [0.25, 0.5, 0.75], 0.1)
print("Approximate Quantiles:", quantiles)

In [0]:
# Approximate distinct count calculation
approx_distinct_count = review_df.select(approx_count_distinct("ProductId", rsd=0.1).alias("approx_distinct_count"))
approx_distinct_count.show()

In [0]:
spark.stop()